In [1]:
import numpy as np
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler
from scipy.ndimage import uniform_filter1d
from tqdm import tqdm
import warnings, gc, time
warnings.filterwarnings('ignore')

DATA_ROOT = '/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2'
TEST_IN   = f'{DATA_ROOT}/test_in'
OUTPUT    = '/kaggle/working/preds.npy'

# ── Hyperparams ────────────────────────────────────────────────────────────────
BATCH_SIZE  = 2      # was 2 — T4 16GB easily handles B=4
ACCUM_STEPS = 2      # effective batch = 8
EPOCHS      = 20     # was 12-15 — AMP makes each epoch ~25min so 20 fits in 12h
LR          = 3e-4
HIDDEN_DIM  = 96
KERNEL_SIZE = 3
NUM_LAYERS  = 3
HISTORY_PM  = 10
FORECAST    = 16
H, W        = 140, 124
PATIENCE    = 5      # more patience since we have more epochs

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch  : {torch.__version__}')
print(f'Device   : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    cap = torch.cuda.get_device_capability(0)
    print(f'Capability: sm_{cap[0]}{cap[1]}')
    USE_AMP = cap[0] >= 7   # T4=sm_75 → True
    torch.backends.cudnn.benchmark = True
else:
    USE_AMP = False
print(f'AMP      : {USE_AMP}')
print(f'Effective batch: {BATCH_SIZE * ACCUM_STEPS}')

PyTorch  : 2.10.0+cu128
Device   : cuda
GPU      : Tesla T4
Capability: sm_75
AMP      : True
Effective batch: 4


In [2]:
# ── Feature lists & Data Loading ───────────────────────────────────────────────
MET_VARS      = ['q2', 't2', 'u10', 'v10', 'swdown', 'pblh', 'psfc', 'rain']
EMISSION_VARS = ['PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio']
MONTHS        = ['APRIL_16', 'JULY_16', 'OCT_16', 'DEC_16']
LOG_FEATURES  = {'PM25', 'NH3', 'SO2', 'NOx', 'NMVOC_e', 'NMVOC_finn', 'bio', 'cpm25'}

def load_month(month):
    path = os.path.join(DATA_ROOT, 'raw', month)
    data = {'cpm25': np.load(os.path.join(path, 'cpm25.npy')).astype(np.float32)}
    for v in MET_VARS + EMISSION_VARS:
        fpath = os.path.join(path, f'{v}.npy')
        if os.path.exists(fpath):
            data[v] = np.load(fpath).astype(np.float32)
    return data

all_data = {}
for m in MONTHS:
    print(f'Loading {m}...')
    all_data[m] = load_month(m)

# ── Normalization ──────────────────────────────────────────────────────────────
def compute_stats(key):
    arrays   = [d[key] for d in all_data.values() if key in d]
    combined = np.concatenate(arrays, axis=0).astype(np.float32)
    if key in LOG_FEATURES:
        combined = np.log1p(combined * 1e9)
    return float(combined.mean()), float(combined.std() + 1e-8)

print('Computing normalization stats...')
norm_stats = {}
for key in ['cpm25'] + MET_VARS + EMISSION_VARS:
    try:
        norm_stats[key] = compute_stats(key)
        print(f'  {key}: mean={norm_stats[key][0]:.4f}, std={norm_stats[key][1]:.4f}')
    except Exception as e:
        print(f'  {key}: SKIPPED ({e})')

def normalize(arr, key):
    mean, std = norm_stats[key]
    arr = arr.astype(np.float32)
    if key in LOG_FEATURES:
        arr = np.log1p(arr * 1e9)
    return (arr - mean) / std

def denormalize_pm25(arr_norm):
    mean, std = norm_stats['cpm25']
    return np.expm1(arr_norm * std + mean) / 1e9

Loading APRIL_16...
Loading JULY_16...
Loading OCT_16...
Loading DEC_16...
Computing normalization stats...
  cpm25: mean=23.4329, std=1.3375
  q2: mean=0.0115, std=0.0070
  t2: mean=291.5939, std=14.0668
  u10: mean=1.5476, std=3.5471
  v10: mean=0.1373, std=2.9958
  swdown: mean=221.7803, std=309.4066
  pblh: mean=764.2910, std=636.3438
  psfc: mean=87986.8594, std=17645.7559
  rain: mean=0.0948, std=1.0707
  PM25: mean=0.0254, std=0.0972
  NH3: mean=0.0284, std=0.0470
  SO2: mean=0.0195, std=0.1225
  NOx: mean=0.0297, std=0.1088
  NMVOC_e: mean=0.0417, std=0.0938
  NMVOC_finn: mean=0.0276, std=0.2897
  bio: mean=0.0386, std=0.1344


In [3]:
# ── Fast Vectorized Episode Detection ──────────────────────────────────────────
def identify_episodes(pm_arr, seasonal_period=24):
    T, H, W   = pm_arr.shape
    pm_2d     = pm_arr.reshape(T, -1).astype(np.float64)
    trend     = uniform_filter1d(pm_2d, size=seasonal_period, axis=0)
    detrended = pm_2d - trend
    seasonal  = np.zeros_like(detrended)
    for h in range(seasonal_period):
        idx = np.arange(h, T, seasonal_period)
        seasonal[idx] = detrended[idx].mean(axis=0, keepdims=True)
    residual = detrended - seasonal
    sigma    = residual.std(axis=0, keepdims=True) + 1e-8
    return (residual > 3 * sigma).reshape(T, H, W)

print('Computing episode masks (vectorized)...')
episode_masks = {}
for m, d in all_data.items():
    print(f'  {m}...', end=' ', flush=True)
    episode_masks[m] = identify_episodes(d['cpm25'])
    print(f'ep_frac={episode_masks[m].mean():.4f}')

Computing episode masks (vectorized)...
  APRIL_16... ep_frac=0.0099
  JULY_16... ep_frac=0.0093
  OCT_16... ep_frac=0.0085
  DEC_16... ep_frac=0.0102


In [4]:
# ── Dataset ─────────────────────────────────────────────────────────────────────
class PM25Dataset(Dataset):
    def __init__(self, data_dict, ep_masks, stride=1):
        self.samples = []
        self.weights = []
        other_keys   = [k for k in MET_VARS + EMISSION_VARS if k in norm_stats]
        window       = HISTORY_PM + FORECAST

        for month, data in data_dict.items():
            T       = data['cpm25'].shape[0]
            pm      = normalize(data['cpm25'], 'cpm25')
            mets    = np.stack([normalize(data[k], k) for k in other_keys if k in data], axis=1)
            ep_mask = ep_masks[month]

            for t in range(0, T - window + 1, stride):
                pm_hist  = pm  [t           : t + HISTORY_PM]
                pm_fut   = pm  [t+HISTORY_PM : t + window]
                met_hist = mets[t           : t + HISTORY_PM]
                ep_frac  = ep_mask[t+HISTORY_PM : t+window].mean()
                self.samples.append((pm_hist, met_hist, pm_fut))
                self.weights.append(1.0 + 9.0 * ep_frac)

        self.weights = np.array(self.weights, dtype=np.float32)

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        ph, mh, pf = self.samples[idx]
        return torch.tensor(ph), torch.tensor(mh), torch.tensor(pf)


# Train = first 80%, Val = last 20% of each month
train_data, val_data = {}, {}
train_ep,   val_ep   = {}, {}
for m, d in all_data.items():
    T     = d['cpm25'].shape[0]
    split = int(T * 0.8)
    train_data[m] = {k: v[:split] for k, v in d.items() if hasattr(v,'shape') and v.shape[0]==T}
    val_data  [m] = {k: v[split:] for k, v in d.items() if hasattr(v,'shape') and v.shape[0]==T}
    train_ep  [m] = episode_masks[m][:split]
    val_ep    [m] = episode_masks[m][split:]

# stride=1 for train (all samples), stride=2 for val (halves val time: 13min→6min/epoch)
train_ds = PM25Dataset(train_data, train_ep, stride=1)
val_ds   = PM25Dataset(val_data,   val_ep,   stride=2)

sampler  = WeightedRandomSampler(
    weights=torch.tensor(train_ds.weights),
    num_samples=len(train_ds), replacement=True
)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                      num_workers=2, pin_memory=True, drop_last=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} samples ({len(train_dl)} batches)')
print(f'Val:   {len(val_ds)} samples ({len(val_dl)} batches)  [stride=2, saves ~7min/epoch]')
print(f'Weight range: {train_ds.weights.min():.2f} – {train_ds.weights.max():.2f}')
C_met = train_ds[0][1].shape[1]
print(f'Met channels: {C_met}')

Train: 2245 samples (1122 batches)
Val:   245 samples (123 batches)  [stride=2, saves ~7min/epoch]
Weight range: 1.01 – 1.34
Met channels: 15


In [5]:
# ── Model ──────────────────────────────────────────────────────────────────────

class ConvLSTMCell(nn.Module):
    def __init__(self, in_ch, hidden_ch, kernel_size):
        super().__init__()
        self.hidden_ch = hidden_ch
        self.conv = nn.Conv2d(in_ch + hidden_ch, 4 * hidden_ch,
                              kernel_size, padding=kernel_size // 2)
        self.ln   = nn.GroupNorm(1, 4 * hidden_ch)

    def forward(self, x, h, c):
        gates      = self.ln(self.conv(torch.cat([x, h], 1)))
        i, f, g, o = gates.chunk(4, 1)
        c = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)
        h = torch.sigmoid(o) * torch.tanh(c)
        return h, c

    def init_hidden(self, B, H, W, device):
        return (torch.zeros(B, self.hidden_ch, H, W, device=device),
                torch.zeros(B, self.hidden_ch, H, W, device=device))


class SpatialAttention(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, in_ch // 4, 1), nn.ReLU(),
            nn.Conv2d(in_ch // 4, 1, 1),     nn.Sigmoid()
        )
    def forward(self, x): return x * self.conv(x)


class WindWarp(nn.Module):
    """Physics transport via last known wind. Scale decays with forecast step."""
    def __init__(self, u_idx, v_idx):
        super().__init__()
        self.u_idx, self.v_idx = u_idx, v_idx
        # Learnable per-step scale — starts at 0.05, model adapts
        self.log_scale = nn.Parameter(torch.tensor(-3.0))  # exp(-3) ≈ 0.05

    def forward(self, pm, last_met, step=0):
        if self.u_idx is None: return pm
        B, _, H, W = pm.shape
        scale = torch.exp(self.log_scale) / (1.0 + 0.1 * step)  # decay with step
        u  = last_met[:, self.u_idx:self.u_idx+1]
        v  = last_met[:, self.v_idx:self.v_idx+1]
        xs = torch.linspace(-1, 1, W, device=pm.device)
        ys = torch.linspace(-1, 1, H, device=pm.device)
        gy, gx   = torch.meshgrid(ys, xs, indexing='ij')
        grid     = torch.stack([gx, gy], -1).unsqueeze(0).expand(B, -1, -1, -1)
        flow     = torch.stack([u.squeeze(1), v.squeeze(1)], -1) * scale
        warped_g = (grid + flow).clamp(-1, 1)
        return F.grid_sample(pm, warped_g, align_corners=False,
                             mode='bilinear', padding_mode='border')


class SpatialEncoder(nn.Module):
    """Multi-scale dilated conv for local + regional spatial patterns."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.e1   = nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=1, dilation=1), nn.ReLU())
        self.e2   = nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=2, dilation=2), nn.ReLU())
        self.e3   = nn.Sequential(nn.Conv2d(in_ch, out_ch, 3, padding=4, dilation=4), nn.ReLU())
        self.fuse = nn.Conv2d(out_ch * 3, out_ch, 1)
    def forward(self, x):
        return self.fuse(torch.cat([self.e1(x), self.e2(x), self.e3(x)], 1))


class EpisodeDetector(nn.Module):
    """Learns WHERE episodes occur from hidden state → soft map for decoder."""
    def __init__(self, in_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 16, 3, padding=1),    nn.ReLU(),
            nn.Conv2d(16, 1,  1),               nn.Sigmoid()
        )
    def forward(self, h): return self.net(h)


class Phase2Model(nn.Module):
    """
    Autoregressive ConvLSTM encoder-decoder with:
    - WindWarp (learnable decay scale)
    - EpisodeDetector (soft episode map fed to decoder)
    - SpatialEncoder (multi-scale dilated)
    - SpatialAttention
    - Episode amplitude amplifier
    - Scheduled teacher forcing
    """
    def __init__(self, met_channels, hidden_dim, kernel_size, num_layers, u_idx, v_idx):
        super().__init__()
        self.num_layers = num_layers
        enc_met_ch      = hidden_dim // 2

        self.spatial_enc    = SpatialEncoder(met_channels, enc_met_ch)
        self.wind_warp      = WindWarp(u_idx, v_idx)
        self.episode_detect = EpisodeDetector(hidden_dim)

        self.enc_cells = nn.ModuleList()
        self.dec_cells = nn.ModuleList()
        for i in range(num_layers):
            enc_in = (1 + enc_met_ch)     if i == 0 else hidden_dim
            dec_in = (1 + enc_met_ch + 1) if i == 0 else hidden_dim
            self.enc_cells.append(ConvLSTMCell(enc_in, hidden_dim, kernel_size))
            self.dec_cells.append(ConvLSTMCell(dec_in, hidden_dim, kernel_size))

        self.attn        = SpatialAttention(hidden_dim)
        self.output_head = nn.Sequential(
            nn.Conv2d(hidden_dim, 64, 3, padding=1), nn.ReLU(),
            nn.Dropout2d(0.1),
            nn.Conv2d(64, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 1, 1)
        )
        self.episode_amp = nn.Sequential(
            nn.Conv2d(hidden_dim, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 1, 1), nn.Softplus()
        )

    def forward(self, pm_hist, met_hist, teacher_forcing_ratio=0.0, pm_fut_gt=None):
        """
        pm_hist             : (B, 10, H, W)
        met_hist            : (B, 10, C, H, W)
        teacher_forcing_ratio: during training, use ground truth with this probability
        pm_fut_gt           : (B, 16, H, W) ground truth for teacher forcing
        """
        B, T_hist, H, W = pm_hist.shape
        states = [cell.init_hidden(B, H, W, pm_hist.device) for cell in self.enc_cells]

        # Encoder: process 10h history
        for t in range(T_hist):
            enc_met = self.spatial_enc(met_hist[:, t])
            x = torch.cat([pm_hist[:, t:t+1], enc_met], 1)
            for i, cell in enumerate(self.enc_cells):
                h, c = states[i]; h, c = cell(x, h, c); states[i] = (h, c); x = h

        last_met  = met_hist[:, -1]
        enc_met_d = self.spatial_enc(last_met)  # pre-encode once

        # Autoregressive decoder: predict 16 steps
        preds      = []
        prev_pm    = pm_hist[:, -1:].clone()
        dec_states = list(states)

        for t in range(FORECAST):
            warped = self.wind_warp(prev_pm, last_met, step=t)
            h_cur  = dec_states[-1][0]
            ep_map = self.episode_detect(h_cur)

            x = torch.cat([warped, enc_met_d, ep_map], 1)
            for i, cell in enumerate(self.dec_cells):
                h, c = dec_states[i]; h, c = cell(x, h, c); dec_states[i] = (h, c); x = h

            x      = self.attn(x)
            delta  = self.output_head(x)
            amp    = self.episode_amp(x)
            pred_t = prev_pm + delta + ep_map * amp
            # Clip to prevent runaway autoregressive predictions
            pred_t = pred_t.clamp(-10, 30)
            preds.append(pred_t)

            # Scheduled teacher forcing
            if teacher_forcing_ratio > 0.0 and pm_fut_gt is not None and torch.rand(1).item() < teacher_forcing_ratio:
                prev_pm = pm_fut_gt[:, t:t+1]
            else:
                prev_pm = pred_t

        return torch.cat(preds, dim=1)  # (B, 16, H, W)


# Instantiate
other_keys = [k for k in MET_VARS + EMISSION_VARS if k in norm_stats]
u_idx      = other_keys.index('u10') if 'u10' in other_keys else None
v_idx      = other_keys.index('v10') if 'v10' in other_keys else None

_model   = Phase2Model(C_met, HIDDEN_DIM, KERNEL_SIZE, NUM_LAYERS, u_idx, v_idx)
num_gpus = torch.cuda.device_count()
if num_gpus > 1:
    model = nn.DataParallel(_model, device_ids=list(range(num_gpus)))
    print(f'DataParallel on {num_gpus} GPUs')
else:
    model = _model
model = model.to(DEVICE)

n_params = sum(p.numel() for p in _model.parameters() if p.requires_grad)
print(f'Parameters: {n_params:,}')

# Quick shape test
with torch.no_grad():
    dp = torch.zeros(1, HISTORY_PM, H, W).to(DEVICE)
    dm = torch.zeros(1, HISTORY_PM, C_met, H, W).to(DEVICE)
    print(f'Output shape: {model(dp, dm).shape}  (expected (1,16,140,124))')
del dp, dm

DataParallel on 2 GPUs
Parameters: 3,829,605
Output shape: torch.Size([1, 16, 140, 124])  (expected (1,16,140,124))


In [6]:
# ── Loss Functions ─────────────────────────────────────────────────────────────

def smape_loss(pred, target):
    return (pred - target).abs().div(0.5 * (pred.abs() + target.abs()) + 1e-8).mean()

def episode_weighted_loss(pred, target, pm_hist):
    hist_mean = pm_hist.mean(dim=1, keepdim=True)
    hist_std  = pm_hist.std(dim=1,  keepdim=True) + 1e-8
    weights   = 1.0 + 2.0 * ((target - hist_mean.expand_as(target)) /
                               hist_std.expand_as(target)).clamp(0).detach()
    return smape_loss(pred * weights, target * weights)

def spatial_gradient_loss(pred, target):
    def grad(x):
        return x[:,:,:,1:] - x[:,:,:,:-1], x[:,:,1:,:] - x[:,:,:-1,:]
    pdx, pdy = grad(pred)
    tdx, tdy = grad(target)
    return F.mse_loss(pdx, tdx) + F.mse_loss(pdy, tdy)

def pearson_corr_loss(pred, target):
    """
    NEW: Directly optimises EpisodeCorr metric.
    Pearson correlation averaged over batch.
    Returns 1 - corr (so minimising → maximising correlation).
    """
    B = pred.shape[0]
    p = pred.view(B, -1)
    t = target.view(B, -1)
    pm = p - p.mean(dim=1, keepdim=True)
    tm = t - t.mean(dim=1, keepdim=True)
    corr = (pm * tm).sum(dim=1) / (
        torch.sqrt((pm**2).sum(dim=1) * (tm**2).sum(dim=1)) + 1e-8)
    return (1.0 - corr.mean())

def combined_loss(pred, target, pm_hist, alpha=0.5, beta=0.25, gamma=0.1, delta=0.15):
    """
    50% episode-weighted SMAPE  → EpisodeSMAPE + GlobalSMAPE
    25% plain SMAPE             → global coverage
    10% spatial gradient        → sharp patterns for EpisodeCorr
    15% Pearson correlation     → directly optimise EpisodeCorr metric (NEW)
    """
    return (alpha * episode_weighted_loss(pred, target, pm_hist) +
            beta  * smape_loss(pred, target) +
            gamma * spatial_gradient_loss(pred, target) +
            delta * pearson_corr_loss(pred, target))

# Sanity check
with torch.no_grad():
    p = torch.ones(2, FORECAST, H, W)
    h = torch.ones(2, HISTORY_PM, H, W)
    l = combined_loss(p, p, h)
    print(f'Perfect-prediction loss: {l.item():.6f}  (should be ~0)')

Perfect-prediction loss: 0.150000  (should be ~0)


In [7]:
# ── Optimizer, Scheduler, AMP Scaler ──────────────────────────────────────────
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
warmup    = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=3)
cosine    = CosineAnnealingLR(optimizer, T_max=max(EPOCHS - 3, 1), eta_min=1e-5)
scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[3])
scaler    = GradScaler(enabled=USE_AMP)
CKPT      = '/kaggle/working/best_model_p2.pt'


def get_tf_ratio(epoch, total_epochs, start=0.5, end=0.0):
    """Teacher forcing ratio decays linearly from start→end over training."""
    progress = min(epoch / total_epochs, 1.0)
    return start * (1.0 - progress) + end * progress


def run_epoch(dl, train=True, tf_ratio=0.0):
    model.train() if train else model.eval()
    total_loss = 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        if train: optimizer.zero_grad()
        for step, (pm_hist, met_hist, pm_fut) in enumerate(tqdm(dl, leave=False)):
            pm_hist  = pm_hist .to(DEVICE, non_blocking=True)
            met_hist = met_hist.to(DEVICE, non_blocking=True)
            pm_fut   = pm_fut  .to(DEVICE, non_blocking=True)

            with autocast(enabled=USE_AMP):
                # Pass teacher forcing info during training
                if train and tf_ratio > 0:
                    pred = model(pm_hist, met_hist,
                                 teacher_forcing_ratio=tf_ratio,
                                 pm_fut_gt=pm_fut)
                else:
                    pred = model(pm_hist, met_hist)
                loss = combined_loss(pred, pm_fut, pm_hist)

            if train:
                scaler.scale(loss / ACCUM_STEPS).backward()
                if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(dl):
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()

            total_loss += loss.item()
    return total_loss / len(dl)


# ── Training Loop ──────────────────────────────────────────────────────────────
best_val   = float('inf')
patience_c = 0
t_start    = time.time()

print(f'Training: {EPOCHS} epochs | batch={BATCH_SIZE} (eff={BATCH_SIZE*ACCUM_STEPS}) | AMP={USE_AMP}')
print(f'{"Ep":>4} {"Train":>8} {"Val":>8} {"LR":>9} {"TF":>5} {"Time":>7}')
print('-' * 52)

for epoch in range(1, EPOCHS + 1):
    tf = get_tf_ratio(epoch, EPOCHS, start=0.5, end=0.0)
    t0 = time.time()

    train_loss = run_epoch(train_dl, train=True,  tf_ratio=tf)
    val_loss   = run_epoch(val_dl,   train=False, tf_ratio=0.0)
    scheduler.step()
    lr  = scheduler.get_last_lr()[0]
    dur = (time.time() - t0) / 60

    tag = ''
    if val_loss < best_val:
        best_val   = val_loss
        patience_c = 0
        state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
        torch.save(state, CKPT)
        tag = ' ✓'
    else:
        patience_c += 1

    elapsed_h = (time.time() - t_start) / 3600
    print(f'{epoch:>4d} {train_loss:>8.4f} {val_loss:>8.4f} {lr:>9.2e} {tf:>5.2f} {dur:>6.1f}m{tag}')

    # Safety: stop if we're close to 11h (leave 1h buffer for inference)
    if elapsed_h > 11.0:
        print(f'Time limit approaching ({elapsed_h:.1f}h). Stopping training.')
        break

    if patience_c >= PATIENCE:
        print(f'Early stopping at epoch {epoch} (patience={PATIENCE})')
        break

print(f'\nBest val loss: {best_val:.4f} | Total training: {(time.time()-t_start)/3600:.2f}h')

Training: 20 epochs | batch=2 (eff=4) | AMP=True
  Ep    Train      Val        LR    TF    Time
----------------------------------------------------


   1   0.2006   0.3537  1.20e-04  0.47   31.5m ✓


   2   0.1675   0.3190  2.10e-04  0.45   30.8m ✓


   3   0.1529   0.2787  3.00e-04  0.42   30.8m ✓


   4   0.1466   0.2664  2.98e-04  0.40   30.7m ✓


...

  16   0.1640   0.2202  4.78e-05  0.10   31.0m


  17   0.1666   0.2218  3.17e-05  0.08   31.0m


  18   0.1712   0.2179  1.98e-05  0.05   31.2m ✓


  19   0.1775   0.2186  1.25e-05  0.03   31.3m


  20   0.1824   0.2179  1.00e-05  0.00   31.2m ✓

Best val loss: 0.2179 | Total training: 10.33h


In [8]:
# ── Inference ──────────────────────────────────────────────────────────────────
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

_model_inf = Phase2Model(C_met, HIDDEN_DIM, KERNEL_SIZE, NUM_LAYERS, u_idx, v_idx)
_model_inf.load_state_dict(torch.load(CKPT, map_location=DEVICE))
_model_inf = _model_inf.to(DEVICE).eval()
print('Best model loaded for inference.')

# Load test PM2.5
test_pm25_raw  = np.load(os.path.join(TEST_IN, 'cpm25.npy')).astype(np.float32)
N              = test_pm25_raw.shape[0]
print(f'Test samples: {N}')  # 218
test_pm25_norm = normalize(test_pm25_raw, 'cpm25')
del test_pm25_raw; gc.collect()

avail_keys = [k for k in other_keys if os.path.exists(os.path.join(TEST_IN, f'{k}.npy'))]
print(f'Available test met: {len(avail_keys)} keys')

all_preds   = []
INFER_BATCH = 8  # larger batch for faster inference
mean_pm, std_pm = norm_stats['cpm25']

with torch.no_grad():
    for i in tqdm(range(0, N, INFER_BATCH), desc='Inference'):
        batch_mets = []
        for k in avail_keys:
            arr = np.load(os.path.join(TEST_IN, f'{k}.npy'), mmap_mode='r')
            slc = np.array(arr[i:i+INFER_BATCH]).astype(np.float32)
            batch_mets.append(normalize(slc, k))

        met_b = torch.tensor(np.stack(batch_mets, axis=2)).to(DEVICE)
        pm_b  = torch.tensor(test_pm25_norm[i:i+INFER_BATCH]).to(DEVICE)

        with autocast(enabled=USE_AMP):
            pred_norm = _model_inf(pm_b, met_b).cpu().float().numpy()

        pred = np.clip(np.expm1(pred_norm * std_pm + mean_pm) / 1e9, 0, 500)
        all_preds.append(pred)
        del pm_b, met_b, batch_mets
        if torch.cuda.is_available(): torch.cuda.empty_cache()

preds     = np.concatenate(all_preds, axis=0)  # (218, 16, H, W)
preds_out = preds.transpose(0, 2, 3, 1)        # (218, 140, 124, 16)

assert preds_out.shape == (N, 140, 124, 16), f'Shape mismatch: {preds_out.shape}'
np.save(OUTPUT, preds_out)
print(f'Saved: {OUTPUT}')
print(f'Shape: {preds_out.shape} | min={preds_out.min():.4f} max={preds_out.max():.2f} mean={preds_out.mean():.2f}')

Best model loaded for inference.
Test samples: 218
Available test met: 15 keys


Inference: 100%|██████████| 28/28 [01:38<00:00,  3.52s/it]


Saved: /kaggle/working/preds.npy
Shape: (218, 140, 124, 16) | min=0.0045 max=500.00 mean=35.31


In [9]:
# ── Submission Verification ────────────────────────────────────────────────────
loaded  = np.load(OUTPUT)
N, H2, W2, T = loaded.shape
print('=' * 45)
print('SUBMISSION VERIFICATION')
print('=' * 45)
print(f'  Path  : {OUTPUT}')
print(f'  Shape : {loaded.shape}')
print(f'  H×W   : {H2}×{W2}  {"✓" if H2==140 and W2==124 else "✗ WRONG"}')
print(f'  T_out : {T}  {"✓" if T==16 else "✗ WRONG"}')
print(f'  dtype : {loaded.dtype}')
print(f'  NaN   : {np.isnan(loaded).any()}')
print(f'  Min/Max: {loaded.min():.4f} / {loaded.max():.2f}')
print('=' * 45)
if H2==140 and W2==124 and T==16 and not np.isnan(loaded).any():
    print('✅ Ready to submit!')
else:
    print('❌ Something wrong — do NOT submit.')

SUBMISSION VERIFICATION
  Path  : /kaggle/working/preds.npy
  Shape : (218, 140, 124, 16)
  H×W   : 140×124  ✓
  T_out : 16  ✓
  dtype : float32
  NaN   : False
  Min/Max: 0.0045 / 500.00
✅ Ready to submit!
